# 02 - Data Cleaning Pipeline

**Goal**: Transform raw postings into a clean, analysis-ready dataset saved as Parquet.
This notebook documents every cleaning decision and its rationale.

**Learning concepts**: Data wrangling, null handling strategies, timestamp parsing,
salary normalization, Parquet format vs CSV.

**Why Parquet?**
- CSV: 493MB, slow to load, no type info, everything is a string on disk
- Parquet: ~100MB (columnar compression), fast to load, preserves dtypes, industry standard

---

In [ ]:
FORCE_RECOMPUTE = False  # Set to True to re-clean from raw CSV even if parquet exists

print(f"Importing libraries...")

import pandas as pd
import numpy as np
from loguru import logger

from talentlens.config import (
    POSTINGS_CSV, PROCESSED_DIR, POSTINGS_CLEAN_PARQUET,
    SALARIES_CSV, HOURS_PER_YEAR,
)
from talentlens.dataset import load_raw_postings, clean_postings, load_secondary_tables

pd.set_option('display.max_columns', None)

print(f"[SUCCESS] Libraries imported successfully.")

## 1. Load the Full Raw Dataset

We load all 3.38M rows. This may take 1-2 minutes depending on your machine.

> **For development**: Set `nrows=50000` to work with a smaller sample first.

In [ ]:
# Set nrows=50000 for faster development; nrows=None for full dataset
# If cleaned parquet exists, we still load raw to show the before/after comparison
df_raw = load_raw_postings(nrows=None)
print(f"Raw shape: {df_raw.shape}")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 2. Understand What We're Dropping (and Why)

Before cleaning, let's see what's missing so we can justify our decisions.

In [3]:
# Null rates for critical columns
critical_cols = ['description', 'title', 'med_salary', 'min_salary', 'max_salary',
                 'remote_allowed', 'formatted_experience_level', 'skills_desc',
                 'applies', 'views', 'listed_time', 'closed_time']

null_report = pd.DataFrame({
    'null_count': df_raw[critical_cols].isna().sum(),
    'null_pct': (df_raw[critical_cols].isna().mean() * 100).round(1),
    'non_null': df_raw[critical_cols].count(),
})
null_report.sort_values('null_pct', ascending=False)

,null_count,null_pct,non_null
closed_time,122776,99.1,1073
skills_desc,121410,98.0,2439
med_salary,117569,94.9,6280
remote_allowed,108603,87.7,15246
applies,100529,81.2,23320
max_salary,94056,75.9,29793
min_salary,94056,75.9,29793
formatted_experience_level,29409,23.7,94440
views,1689,1.4,122160
title,0,0.0,123849


In [4]:
# How many duplicate job_ids?
dupes = df_raw.duplicated(subset=['job_id'], keep=False)
print(f"Duplicate job_id rows: {dupes.sum():,} ({dupes.mean()*100:.1f}%)")
print(f"Unique job_ids: {df_raw['job_id'].nunique():,}")

Duplicate job_id rows: 0 (0.0%)
Unique job_ids: 123,849


## 3. Run the Cleaning Pipeline

Our `clean_postings()` function (from `talentlens/dataset.py`) performs:

1. **Drop null descriptions** — can't do NLP without text
2. **Drop null titles** — need titles for classification
3. **Deduplicate on job_id** — keep first occurrence
4. **Parse timestamps** — convert Unix ms → datetime
5. **Normalize salaries** — hourly × 2,080 → yearly
6. **Create derived columns** — `is_remote`, `days_open`, `experience_level`

In [ ]:
if not FORCE_RECOMPUTE and POSTINGS_CLEAN_PARQUET.exists():
    df_clean = pd.read_parquet(POSTINGS_CLEAN_PARQUET)
    print(f"Loaded cleaned data from cache ({POSTINGS_CLEAN_PARQUET.name})")
else:
    df_clean = clean_postings(df_raw)

print(f"\n{'='*50}")
print(f"  CLEANING SUMMARY")
print(f"{'='*50}")
print(f"  Raw rows:     {len(df_raw):>12,}")
print(f"  Clean rows:   {len(df_clean):>12,}")
print(f"  Rows dropped: {len(df_raw) - len(df_clean):>12,} ({(1 - len(df_clean)/len(df_raw))*100:.1f}%)")
print(f"{'='*50}")
print(f"\n>>> UPDATE YOUR RESUME WITH THIS NUMBER: {len(df_clean):,} job postings <<<")

## 4. Validate the Cleaned Data

In [6]:
# Sanity checks
assert df_clean['description'].isna().sum() == 0, "Null descriptions found!"
assert df_clean['title'].isna().sum() == 0, "Null titles found!"
assert df_clean['job_id'].is_unique, "Duplicate job_ids found!"
assert 'is_remote' in df_clean.columns, "Missing is_remote column!"
assert 'experience_level' in df_clean.columns, "Missing experience_level column!"
assert 'med_salary_yearly' in df_clean.columns, "Missing med_salary_yearly column!"

print("All validation checks passed!")
print(f"\nCleaned columns ({len(df_clean.columns)}):")
for col in sorted(df_clean.columns):
    print(f"  {col}: {df_clean[col].dtype}")

All validation checks passed!

Cleaned columns (37):
  application_type: object
  application_url: object
  applies: float64
  closed_time: datetime64[ns]
  company_id: float64
  company_name: object
  compensation_type: object
  currency: object
  days_open: float64
  description: object
  experience_level: object
  expiry: datetime64[ns]
  fips: float64
  formatted_experience_level: object
  formatted_work_type: object
  is_remote: bool
  job_id: int64
  job_posting_url: object
  listed_time: datetime64[ns]
  location: object
  max_salary: float64
  max_salary_yearly: float64
  med_salary: float64
  med_salary_yearly: float64
  min_salary: float64
  min_salary_yearly: float64
  normalized_salary: float64
  original_listed_time: datetime64[ns]
  pay_period: object
  posting_domain: object
  remote_allowed: float64
  skills_desc: object
  sponsored: float64
  title: object
  views: float64
  work_type: object
  zip_code: object


## 5. Inspect Key Derived Columns

In [7]:
# Experience level distribution
print("=== Experience Levels ===")
print(df_clean['experience_level'].value_counts())

print("\n=== Remote Work ===")
print(df_clean['is_remote'].value_counts())

print("\n=== Salary Coverage ===")
has_salary = df_clean['med_salary_yearly'].notna()
print(f"Postings with salary: {has_salary.sum():,} ({has_salary.mean()*100:.1f}%)")

print("\n=== Salary Stats (yearly, where available) ===")
print(df_clean['med_salary_yearly'].describe().round(0))

print("\n=== Days Open Stats ===")
if 'days_open' in df_clean.columns:
    print(df_clean['days_open'].describe().round(1))

=== Experience Levels ===
experience_level
Mid-Senior level    41489
Entry level         36708
Unknown             29402
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

=== Remote Work ===
is_remote
False    108599
True      15243
Name: count, dtype: int64

=== Salary Coverage ===
Postings with salary: 6,280 (5.1%)

=== Salary Stats (yearly, where available) ===
count       6280.0
mean       63121.0
std        52417.0
min            0.0
25%        37440.0
50%        47840.0
75%        72800.0
max      1248000.0
Name: med_salary_yearly, dtype: float64

=== Days Open Stats ===
count    1071.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: days_open, dtype: float64


## 6. Save the Cleaned Dataset

In [ ]:
# Save as Parquet (much smaller and faster than CSV)
if not FORCE_RECOMPUTE and POSTINGS_CLEAN_PARQUET.exists():
    print(f"Clean parquet already saved at {POSTINGS_CLEAN_PARQUET.name}")
else:
    # Ensure output directory exists
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_clean.to_parquet(POSTINGS_CLEAN_PARQUET, index=False)
    print(f"Saved cleaned data to {POSTINGS_CLEAN_PARQUET}")

# Compare file sizes
import os
csv_size = os.path.getsize(POSTINGS_CSV) / 1e6
parquet_size = os.path.getsize(POSTINGS_CLEAN_PARQUET) / 1e6

print(f"Raw CSV size:       {csv_size:>8.1f} MB")
print(f"Clean Parquet size: {parquet_size:>8.1f} MB")
print(f"Compression ratio:  {csv_size/parquet_size:.1f}x smaller")

# Verify we can reload it
df_verify = pd.read_parquet(POSTINGS_CLEAN_PARQUET)
assert len(df_verify) == len(df_clean), "Row count mismatch!"
print(f"\nVerified: {len(df_verify):,} rows saved and reloaded successfully.")

## Cleaning Decisions Log

| Decision | Rationale | Rows Affected |
|----------|-----------|---------------|
| Drop null descriptions | Can't do NLP/RAG without text | *(fill in)* |
| Drop null titles | Need titles for job classification | *(fill in)* |
| Deduplicate on job_id | Same posting listed multiple times | *(fill in)* |
| Convert hourly → yearly salary | Need consistent units for comparison | N/A (transform) |
| Parse timestamps to datetime | Enable time-based analysis | N/A (transform) |

---

**→ Next notebook**: `03-mp-exploratory-data-analysis.ipynb`